<a href="https://colab.research.google.com/github/huyle2411-hub/credit-risk-scorecard/blob/main/notebooks/04_challenger_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Scorecard — Notebook 04: Boosting challenger + SQL duckdb

Nội dung: (1) so scorecard logistic với một boosting challenger, (2) một truy vấn SQL giám sát bad-rate theo score band.

In [ ]:
!pip install optbinning duckdb -q
import pandas as pd, numpy as np
from optbinning import BinningProcess
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
import statsmodels.api as sm, duckdb
pd.set_option('display.max_columns', None)

Cell dưới đây dựng lại mô hình scorecard Logistic Regression từ NB02 và NB03 để notebook này có thể chạy độc lập. Đồng thời giữ lại bộ dữ liệu đã làm sạch nhưng chưa WOE để huấn luyện mô hình challenger trên cùng một nguồn dữ liệu.

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('cs-training.csv', index_col=0)
target = 'SeriousDlqin2yrs'
late = ['NumberOfTime30-59DaysPastDueNotWorse','NumberOfTimes90DaysLate','NumberOfTime60-89DaysPastDueNotWorse']

# --- cleaning (nhu NB02) ---
d = df.copy()
d['severe_delinq_flag'] = (d[late] >= 90).any(axis=1).astype(int)
d[late] = d[late].mask(d[late] >= 90)
d['age'] = d['age'].replace(0, np.nan)
for c in ['RevolvingUtilizationOfUnsecuredLines','DebtRatio']:
    d[c] = d[c].clip(upper=d[c].quantile(0.99))
d['NumberOfDependents'] = d['NumberOfDependents'].fillna(0)

y = d[target].values
X_raw = d.drop(columns=[target])

idx = np.arange(len(d))
i_tr, i_te = train_test_split(idx, test_size=0.3, stratify=y, random_state=42)

# --- logistic tren WOE (nhu NB03) ---
bp = BinningProcess(variable_names=list(X_raw.columns)); bp.fit(X_raw.values, y)
Xw = pd.DataFrame(bp.transform(X_raw.values), columns=X_raw.columns, index=X_raw.index)
iv = bp.summary().set_index('name')['iv']
Xw = Xw[[c for c in X_raw.columns if iv[c] >= 0.02]]
logit = sm.Logit(y[i_tr], sm.add_constant(Xw.iloc[i_tr])).fit(disp=0)
p_lg = logit.predict(sm.add_constant(Xw.iloc[i_te]))
auc_lg = roc_auc_score(y[i_te], p_lg)
print('Logistic scorecard (WOE)  AUC = %.4f' % auc_lg)

##1. Boosting challenger model

Challenger là mô hình dùng để đối chiếu với mô hình chính nhằm đánh giá liệu còn dư địa cải thiện hiệu năng hay không. Chọn Gradient Boosting vì mô hình này tự học được các quan hệ phi tuyến và tương tác giữa các biến, đồng thời xử lý trực tiếp giá trị thiếu nên không cần biến đổi WOE. Kết quả của challenger đóng vai trò như một "mốc tham chiếu" để đánh giá Logistic Regression còn bỏ lỡ bao nhiêu khả năng dự báo.

In [ ]:
gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_raw.iloc[i_tr], y[i_tr])
p_gb = gb.predict_proba(X_raw.iloc[i_te])[:, 1]
auc_gb = roc_auc_score(y[i_te], p_gb)

print('Logistic (WOE)      AUC = %.4f' % auc_lg)
print('HistGradBoost       AUC = %.4f' % auc_gb)
print('Boosting hon         = %+.4f AUC' % (auc_gb - auc_lg))

Gradient Boosting đạt AUC cao hơn khoảng 0.013 so với Logistic Regression ngay cả khi chưa tinh chỉnh tham số, cho thấy vẫn có thể khai thác thêm một phần tín hiệu trong dữ liệu. Tuy nhiên, mức cải thiện này tương đối nhỏ. Trong bối cảnh scorecard ngân hàng, khả năng giải thích mô hình, đưa ra lý do từ chối tín dụng và đáp ứng yêu cầu của cơ quan quản lý quan trọng hơn một mức tăng nhỏ về hiệu năng. Vì vậy, Logistic Regression vẫn là lựa chọn phù hợp để triển khai, trong khi Gradient Boosting được giữ làm mô hình benchmark để theo dõi và so sánh. Điều này không có nghĩa Logistic Regression yếu, mà phản ánh sự cân bằng giữa hiệu năng và khả năng giải thích trong bài toán chấm điểm tín dụng.

##2. SQL duckdb

Phần SQL mô phỏng một tác vụ thường gặp trong giám sát danh mục tín dụng: chia khách hàng thành các score band và tính bad rate của từng nhóm. Đây là cách đội quản trị rủi ro và chất lượng tín dụng theo dõi hiệu quả của scorecard trong các kỳ vận hành.

In [ ]:
factor = 20 / np.log(2); offset = 600 - factor * np.log(50)
score = offset + factor * np.log((1 - p_lg) / np.clip(p_lg, 1e-9, 1))
portfolio = pd.DataFrame({'score': score.values, 'actual': y[i_te]})

query = '''
SELECT
  CASE WHEN score < 500 THEN '1. <500'
       WHEN score < 550 THEN '2. 500-550'
       WHEN score < 600 THEN '3. 550-600'
       ELSE '4. 600+' END          AS score_band,
  COUNT(*)                          AS n_accounts,
  ROUND(AVG(actual) * 100, 2)       AS bad_rate_pct
FROM portfolio
GROUP BY score_band
ORDER BY score_band
'''
duckdb.query(query).to_df()

Kết quả cho thấy bad rate giảm đều theo score band, từ khoảng 51% ở nhóm điểm thấp xuống còn khoảng 1% ở nhóm điểm cao, cho thấy scorecard đã phân tách rủi ro hiệu quả. Các score band này là cơ sở để thiết lập cutoff phê duyệt khoản vay, đồng thời hỗ trợ ước lượng bad rate của toàn danh mục trong quá trình vận hành. Với kết quả trên, dự án đã hoàn thiện quy trình xây dựng một mô hình PD (Probability of Default) theo hướng tiếp cận của các ngân hàng thương mại, từ xử lý dữ liệu, xây dựng scorecard, đánh giá hiệu năng đến mô phỏng triển khai và giám sát bằng SQL.